# LLMのAPIの基礎


In [1]:
!python --version

Python 3.11.2


In [2]:
print("hello")

hello


## Chat Completions API を試してみよう！


### OpenAI API キー（環境変数）の設定


In [3]:
# このコードを実行する前に...
# `.env.template` ファイルをコピーして `.env` ファイルを作成してください。
# `.env` ファイルには OpenAI API キーを記載してください。

from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

True

In [4]:
import os

print(os.environ["OPENAI_API_KEY"][:3])
# "sk-" と表示されれば、OpenAIのAPIキーを環境変数に設定できています

sk-


### Chat Completions API の呼び出し


In [5]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"},
    ],
    reasoning_effort="minimal",  # この指定については後ほど解説します
)
print(response.to_json(indent=2))

{
  "id": "chatcmpl-DgP32MaO0Gn9bkILOm66AVfuvMndK",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "こんにちは、ジョンさん！はじめまして。今日はどんなことをお手伝いできますか？もし自己紹介や話したいトピックがあれば教えてください。",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1778998432,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 48,
    "prompt_tokens": 24,
    "total_tokens": 72,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}


### 会話履歴を踏まえた応答を得る


In [6]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"},
        {
            "role": "assistant",
            "content": "こんにちは、ジョンさん！お会いできて嬉しいです。今日はどんなことをお話ししましょうか？",
        },
        {"role": "user", "content": "私の名前が分かりますか？"},
    ],
    reasoning_effort="minimal",
)
print(response.to_json(indent=2))

{
  "id": "chatcmpl-DgP34qPEd6JDkTnWBG8woASxg7NBJ",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "はい、ジョンさんのお名前は会話の中で教えていただいたので分かります。もし別のお名前やニックネームを使いたい場合は教えてください。何についてお話ししましょうか？",
        "refusal": null,
        "role": "assistant",
        "annotations": []
      }
    }
  ],
  "created": 1778998434,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 59,
    "prompt_tokens": 70,
    "total_tokens": 129,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}


### ストリーミングで応答を得る


In [7]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"},
    ],
    stream=True,
    reasoning_effort="minimal",
)

for chunk in response:
    if len(chunk.choices) == 0:
        continue
    content = chunk.choices[0].delta.content
    if content is not None:
        print(content, end="", flush=True)

こんにちは、ジョンさん！はじめまして。今日はどんなことをお手伝いできますか？

## Vision（画像入力）


In [8]:
from openai import OpenAI

client = OpenAI()

image_url = "https://raw.githubusercontent.com/GenerativeAgents/agent-book/refs/heads/main/assets/cover.jpg"

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "画像を説明してください。"},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        }
    ],
    reasoning_effort="minimal",
)

print(response.choices[0].message.content)

この画像は、技術系の書籍の表紙の一部のようです。特徴は次のとおりです。

- 上部には大きな黒字の日本語/英語のタイトル風テキストが並んでいます。「LangChainとLangGraphによる RAG・AIージェント [実践] 入門」などのフレーズが目立ちます。フォントは力強く太字で、左から右へ配置されています。
- 複数の色使い：背景はクリーム色、見出し部分の文字は黒と青（RAG・AIエージェントの部分が特に大きな青色）、実践の部分は黒い括弧付きのデザインになっています。
- 真ん中ほどには大きな鳥の絵（赤いオウムのような鳥）が横向きに広がっており、青い背景と対照しています。鳥の翼の動きがダイナミックで、デザインのアクセントになっています。
- 下部には、白い背景に黒い日本語の長い本文テキストが並んでいます。箇条書きの導入として赤い点のリストがいくつか見え、右下には小さなイラストと「エンジニア選書」などのロゴ風要素があります。

全体として、技術書のカバーデザインで、LangChain/LangGraph、RAG、AIエージェントに関する実践的解説を示唆する印象です。


## LangChainの「Model」


### LangChainでのLLMの呼び出し


In [9]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gpt-5-nano",
    model_provider="openai",
    reasoning_effort="minimal",
)

messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("こんにちは！私はジョンと言います"),
    AIMessage("こんにちは、ジョンさん！どのようにお手伝いできますか？"),
    HumanMessage("私の名前がわかりますか？"),
]

ai_message = model.invoke(messages)
print(ai_message.content)

はい、前のメッセージで「ジョンさん」とおっしゃっていたので、名前はジョンだと理解しています。もし正式名や別の呼び方があれば教えてください。あなたの名前を覚えておくこともできます。どうしますか？


### ストリーミング


In [10]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gpt-5-nano",
    model_provider="openai",
    reasoning_effort="minimal",
)

messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("こんにちは！"),
]

for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

こんにちは！どうお手伝いしましょうか？

# プロンプトの構成要素の基礎


## Few-shot プロンプティング


In [11]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "developer",
            "content": "入力がAIに関係するかどうかを判断して「ＴＲＵＥ」または「ＦＡＬＳＥ」で端的に回答してください。",
        },
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
    reasoning_effort="minimal",
)
print(response.choices[0].message.content)

TRUE


In [12]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "developer",
            "content": "入力がAIに関係するかどうかを判断して「ＴＲＵＥ」または「ＦＡＬＳＥ」で端的に回答してください。",
        },
        {"role": "user", "content": "AIの進化はすごい"},
        {"role": "assistant", "content": "ＴＲＵＥ"},
        {"role": "user", "content": "今日は良い天気だ"},
        {"role": "assistant", "content": "ＦＡＬＳＥ"},
        {"role": "user", "content": "LLMの進化は早い"},
        {"role": "assistant", "content": "ＴＲＵＥ"},
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
    reasoning_effort="minimal",
)
print(response.choices[0].message.content)

ＴＲＵＥ


## Zero-shot Chain of Thought プロンプティング


In [13]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "回答だけ一言で出力してください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
    reasoning_effort="minimal",
)
print(response.choices[0].message.content)

計算結果は4です。


In [14]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "ステップバイステップで考えてください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
    reasoning_effort="minimal",
)
print(response.choices[0].message.content)

ステップ1: 優先順位を確認
- 乗算は加減算より優先される。

ステップ2: 先に計算する部分を取り出す
- 2 * 3 = 6
- 4 * 2 = 8

ステップ3: 置き換えて式を再構成
- 元の式は 10 + 2*3 - 4*2 なので 10 + 6 - 8

ステップ4: 加算・減算を左から計算
- 10 + 6 = 16
- 16 - 8 = 8

最終結果: 8


## Reasoning モデル


In [15]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "回答だけ一言で出力してください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
    reasoning_effort="high",  # Reasoningを最大に
)
print(response.choices[0].message.content)

8
